### Imports

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

## Base Path

In [0]:
# Base Paths
SILVER_BASE_PATH = "abfss://silver@stretaillakehouse01.dfs.core.windows.net/"
GOLD_BASE_PATH   = "abfss://gold@stretaillakehouse01.dfs.core.windows.net/"

# Source Path
SILVER_PRODUCT_PATH = SILVER_BASE_PATH + "products/"

# Target Path
GOLD_DIM_PRODUCT_PATH = GOLD_BASE_PATH + "dim_product/"

### Reading from Silver Layer

In [0]:
df_dim_product = (
    spark.read
         .format("parquet")
         .load(SILVER_PRODUCT_PATH)
)

In [0]:
df_dim_product.printSchema()

df_dim_product.show(5, truncate=False)

print(f"Total Rows: {df_dim_product.count()}")

root
 |-- Product_ID: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Product_SubCategory: string (nullable = true)
 |-- Product_Brand: string (nullable = true)
 |-- Product_Price: double (nullable = true)
 |-- Product_Unit: string (nullable = true)
 |-- Product_Launch_Date: date (nullable = true)

+----------+---------------------------+----------------+-------------------+-------------+-------------+------------+-------------------+
|Product_ID|Product_Name               |Product_Category|Product_SubCategory|Product_Brand|Product_Price|Product_Unit|Product_Launch_Date|
+----------+---------------------------+----------------+-------------------+-------------+-------------+------------+-------------------+
|P0001     |Wildcraft Backpack         |Fashion         |Wildcraft          |Wildcraft    |47511.43     |Each        |2024-01-22         |
|P0002     |Nike Sports T-Shirt        |Fashion         |Nike         

### Null Value Verification

In [0]:
df_dim_product.select(
    [
        sum(col(c).isNull().cast("int")).alias(c)
        for c in df_dim_product.columns
    ]
).show()

+----------+------------+----------------+-------------------+-------------+-------------+------------+-------------------+--------------+
|Product_ID|Product_Name|Product_Category|Product_SubCategory|Product_Brand|Product_Price|Product_Unit|Product_Launch_Date|Price_Category|
+----------+------------+----------------+-------------------+-------------+-------------+------------+-------------------+--------------+
|         0|           0|               0|                  0|           14|            0|           0|                  0|             0|
+----------+------------+----------------+-------------------+-------------+-------------+------------+-------------------+--------------+



In [0]:
df_dim_product.filter("Product_Brand IS NULL").show()

+----------+--------------------+--------------------+-------------------+-------------+-------------+------------+-------------------+--------------+
|Product_ID|        Product_Name|    Product_Category|Product_SubCategory|Product_Brand|Product_Price|Product_Unit|Product_Launch_Date|Price_Category|
+----------+--------------------+--------------------+-------------------+-------------+-------------+------------+-------------------+--------------+
|     P0090|Car Charger Fast 30W| Vehicle Accessories|                Car|         NULL|     19830.43|        Each|         2025-08-06|      Standard|
|     P0204| Bajaj Mixer Grinder|      Home & Kitchen|              Bajaj|         NULL|      7645.22|         Set|         2022-07-25|        Budget|
|     P0211|     Nivea Face Wash|Beauty & Personal...|              Nivea|         NULL|     22529.35|        Each|         2024-03-21|      Standard|
|     P0263|     Bike Body Cover| Vehicle Accessories|               Bike|         NULL|      

## Observation:
### Product_Brand has 14 NULL values.
### No imputation performed as no business rule or trusted reference data is available.

### Duplicate Value Verification

In [0]:
df_dim_product = df_dim_product.groupBy("Product_ID").count()
df_dim_product = df_dim_product.filter(col("count") > 1)
df_dim_product.display()

Product_ID,count


### Validation for Derived Column

In [0]:
df_dim_product.select("Product_Price").summary().show()

+-------+------------------+
|summary|     Product_Price|
+-------+------------------+
|  count|              1000|
|   mean| 24851.86617000004|
| stddev|14560.094763853254|
|    min|            244.81|
|    25%|          12369.98|
|    50%|          24214.88|
|    75%|          37775.67|
|    max|          49959.61|
+-------+------------------+



### Creating Derived Column Price_Category

In [0]:
df_dim_product = (
    df_dim_product
    .withColumn(
        "Price_Category",
        when(col("Product_Price") <= 12369.98, "Budget")
        .when(col("Product_Price") <= 24214.88, "Standard")
        .when(col("Product_Price") <= 37775.67, "Premium")
        .otherwise("Luxury")
    )
)

In [0]:
df_product_validation = df_dim_product.select("Product_Name","Product_Price","Price_Category")
display(df_product_validation.limit(10))

Product_Name,Product_Price,Price_Category
Wildcraft Backpack,47511.43,Luxury
Nike Sports T-Shirt,34977.22,Premium
Prestige Pressure Cooker 5L,49429.51,Luxury
Tata Tea Gold,24461.77,Premium
Yoga Mat Pro,43225.65,Luxury
Milton Water Bottle,13875.58,Standard
Boat Rockerz 450,30877.01,Premium
Wildcraft Backpack,15271.17,Standard
Cello Storage Container,7090.75,Budget
Studds Full Face Helmet,19247.52,Standard


In [0]:
df_dim_product.groupBy("Price_Category").count().show()

+--------------+-----+
|Price_Category|count|
+--------------+-----+
|       Premium|  250|
|      Standard|  250|
|        Budget|  250|
|        Luxury|  250|
+--------------+-----+



In [0]:
(
    df_dim_product.write
    .mode("overwrite")
    .format("parquet")
    .save(GOLD_DIM_PRODUCT_PATH)
)

In [0]:
df_dim_gold_product = (
    spark.read
         .format("parquet")
         .load(GOLD_DIM_PRODUCT_PATH)
)
df_dim_gold_product.printSchema()

df_dim_gold_product.show(5, truncate=False)


root
 |-- Product_ID: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Product_SubCategory: string (nullable = true)
 |-- Product_Brand: string (nullable = true)
 |-- Product_Price: double (nullable = true)
 |-- Product_Unit: string (nullable = true)
 |-- Product_Launch_Date: date (nullable = true)
 |-- Price_Category: string (nullable = true)

+----------+---------------------------+----------------+-------------------+-------------+-------------+------------+-------------------+--------------+
|Product_ID|Product_Name               |Product_Category|Product_SubCategory|Product_Brand|Product_Price|Product_Unit|Product_Launch_Date|Price_Category|
+----------+---------------------------+----------------+-------------------+-------------+-------------+------------+-------------------+--------------+
|P0001     |Wildcraft Backpack         |Fashion         |Wildcraft          |Wildcraft    |47511.43     |Each        |